# 02 · Feature check

Run `features.extract` over the whole dataset. Two questions: is it fast enough for an API, and does any single feature act as a shortcut the way the raw URL format did (99.56%)?

In [1]:
import time

import pandas as pd
from sklearn.metrics import roc_auc_score

from phishing.data import load
from phishing.features import NUMERIC_FEATURES, extract_many

df = load()
start = time.perf_counter()
X = extract_many(df["url"])
elapsed = time.perf_counter() - start
print(f"{len(df):,} URLs in {elapsed:.1f}s, {elapsed / len(df) * 1e6:.0f} µs per URL")

235,795 URLs in 19.6s, 83 µs per URL


Well under a millisecond per URL, so the API won't notice it.

In [2]:
y = df["is_phishing"]
rows = []
for col in NUMERIC_FEATURES:
    auc = roc_auc_score(y, X[col])
    rows.append(
        {
            "feature": col,
            "auc_alone": max(auc, 1 - auc),
            "median_legit": X.loc[y == 0, col].median(),
            "median_phishing": X.loc[y == 1, col].median(),
        }
    )
pd.DataFrame(rows).sort_values("auc_alone", ascending=False).round(3)

,feature,auc_alone,median_legit,median_phishing
1,n_labels,0.720,2.000,3.000
0,host_length,0.693,15.000,20.000
7,entropy,0.687,3.337,3.607
3,n_digits,0.670,0.000,0.000
4,digit_ratio,0.668,0.000,0.000
2,n_hyphens,0.623,0.000,0.000
10,on_free_host,0.580,0.000,0.000
5,longest_label,0.571,10.000,11.000
6,longest_consonant_run,0.565,2.000,2.000
8,is_ip,0.503,0.000,0.000


AUC of one feature on its own (0.5 = useless, 1.0 = perfect). The best is `n_labels` at about 0.72, so nothing here is a shortcut: a model has to combine them. That's the healthy version of this table.

`is_ip` and `has_punycode` are near 0.5 because they're rare (punycode is 154 rows), not because they're useless. When they fire, they're almost always phishing:

In [3]:
for col in ["is_ip", "has_punycode", "on_free_host"]:
    hit = X[col] == 1
    print(f"{col:14s} rows={hit.sum():6d}  phishing share={y[hit].mean():.3f}")

is_ip          rows=   608  phishing share=1.000
has_punycode   rows=   154  phishing share=1.000
on_free_host   rows= 16182  phishing share=1.000


All three are 100% phishing here. For `is_ip` and `has_punycode` that's plausible in real life too. For `on_free_host` it's the Day 2 bias again (the legit side has no free-hosted sites at all), so a model will lean on it harder than it should. Day 6 tests it on real legit `github.io` / `vercel.app` sites.

In [4]:
X["tld"].nunique(), X["tld"].value_counts().head(10).to_dict()

(570,
 {'com': 112565,
  'org': 18793,
  'net': 7097,
  'app': 6508,
  'uk': 6395,
  'co': 5422,
  'io': 4201,
  'de': 3996,
  'ru': 3875,
  'au': 2979})

570 distinct TLDs. Day 4 will need to group rare ones into "other" before one-hot encoding.

Dead end: I tried `tldextract` to get the registered domain (so `bbc.co.uk` → TLD `co.uk`), but installing it hung on this network. For now the TLD is the last label (`uk`). It's needed properly for the grouped train/test split on Day 4.